# Deep Learning Diagnostics Lab

작은 CNN 하나를 학습시키면서 학습 동역학 → 표현 구조 → loss landscape를 순서대로 관찰합니다.

이번 버전은 CIFAR-10을 `torchvision` 원본 서버에서 직접 받지 않고 **Hugging Face Hub (`uoft-cs/cifar10`)** 에서 받습니다.


## 0. 환경 설정

`FAST_MODE=True`이면 CIFAR-10 일부와 6 epoch만 사용합니다.


In [ ]:
!pip -q install datasets umap-learn tensorboard

import json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap
from google.colab import drive

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED=7
FAST_MODE=True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/deep_learning_diagnostics')
CSV_DIR=DRIVE_ROOT/'csv'; NPZ_DIR=DRIVE_ROOT/'npz'; FIG_DIR=DRIVE_ROOT/'figures'; TB_DIR=DRIVE_ROOT/'tensorboard'; SUMMARY_DIR=DRIVE_ROOT/'summaries'
LOCAL_CKPT_DIR=Path('/content/local_checkpoints')
for folder in [CSV_DIR,NPZ_DIR,FIG_DIR,TB_DIR,SUMMARY_DIR,LOCAL_CKPT_DIR]: folder.mkdir(parents=True,exist_ok=True)
print('PyTorch version:',torch.__version__)
print('Device:',DEVICE)


## 1. 데이터와 진단 시점 준비

Hugging Face CDN에서 CIFAR-10을 받습니다. 학습용 입력에는 crop/flip을 적용하고, CKA/PCA처럼 epoch 사이 표현을 비교하는 진단용 입력에는 augmentation을 적용하지 않습니다.

- 6 epochs → `0, 2, 4, 6`
- 12 epochs → `0, 3, 6, 9, 12`


In [ ]:
mean=(0.4914,0.4822,0.4465); std=(0.2470,0.2435,0.2616)
train_transform=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize(mean,std)])

print('Loading CIFAR-10 from Hugging Face Hub...')
hf_train=load_dataset('uoft-cs/cifar10',split='train')
print('HF cache ready:',len(hf_train),'samples')

class CIFAR10FromHF(Dataset):
    def __init__(self,hf_dataset,transform): self.ds=hf_dataset; self.transform=transform
    def __len__(self): return len(self.ds)
    def __getitem__(self,index):
        row=self.ds[index]
        return self.transform(row['img'].convert('RGB')), int(row['label'])

train_aug=CIFAR10FromHF(hf_train,train_transform)
train_eval=CIFAR10FromHF(hf_train,eval_transform)
permutation=torch.randperm(len(train_aug),generator=torch.Generator().manual_seed(SEED)).tolist()
if FAST_MODE: n_train,n_val,EPOCHS=12000,2000,6
else: n_train,n_val,EPOCHS=40000,5000,12
train_indices=permutation[:n_train]; val_indices=permutation[n_train:n_train+n_val]
train_dataset=Subset(train_aug,train_indices); train_eval_dataset=Subset(train_eval,train_indices); val_dataset=Subset(train_eval,val_indices)
loader_args=dict(batch_size=256,num_workers=2,pin_memory=torch.cuda.is_available())
train_loader=DataLoader(train_dataset,shuffle=True,**loader_args); train_eval_loader=DataLoader(train_eval_dataset,shuffle=False,**loader_args); val_loader=DataLoader(val_dataset,shuffle=False,**loader_args)
diag_interval=max(1,round(EPOCHS/4)); DIAG_EPOCHS=sorted(set([0]+list(range(diag_interval,EPOCHS+1,diag_interval))+[EPOCHS]))
print('Train samples:',len(train_dataset)); print('Validation samples:',len(val_dataset)); print('Diagnostic epochs:',DIAG_EPOCHS)


## 2. 진단할 CNN 정의

`image → stem → block1 → block2 → penultimate → head → logits`


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU())
        self.block1=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.block2=nn.Sequential(nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.pool=nn.AdaptiveAvgPool2d(1); self.penultimate=nn.Linear(128,64); self.head=nn.Linear(64,10)
    def forward(self,x,return_features=False):
        features={}
        x=self.stem(x); features['stem']=x.mean((2,3))
        x=self.block1(x); features['block1']=x.mean((2,3))
        x=self.block2(x); features['block2']=x.mean((2,3))
        x=self.pool(x).flatten(1); x=F.relu(self.penultimate(x)); features['penultimate']=x
        logits=self.head(x)
        return (logits,features) if return_features else logits

LAYER_NAMES=['stem','block1','block2','penultimate']
TRACKED_PARAMETERS={'stem':'stem.0.weight','block1':'block1.0.weight','block2':'block2.0.weight','penultimate':'penultimate.weight','head':'head.weight'}
torch.manual_seed(SEED); INITIAL_STATE={k:v.cpu().clone() for k,v in SmallCNN().state_dict().items()}


## 3. 평가와 학습 동역학

gradient norm과 update-to-weight를 batch마다 기록합니다.


In [ ]:
@torch.no_grad()
def evaluate(model,loader):
    model.eval(); loss_sum=correct=count=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); logits=model(x)
        loss_sum+=F.cross_entropy(logits,y,reduction='sum').item(); correct+=(logits.argmax(1)==y).sum().item(); count+=y.numel()
    return loss_sum/count,correct/count

def train_model(run_name,optimizer_name):
    model=SmallCNN().to(DEVICE); model.load_state_dict(INITIAL_STATE)
    optimizer=torch.optim.SGD(model.parameters(),lr=0.08,momentum=0.9) if optimizer_name=='sgd' else torch.optim.AdamW(model.parameters(),lr=2e-3)
    writer=SummaryWriter(str(TB_DIR/run_name)); history=[]; dynamics=[]; global_step=0
    torch.save(model.state_dict(),LOCAL_CKPT_DIR/f'{run_name}_epoch0.pt')
    for epoch in range(1,EPOCHS+1):
        model.train(); train_loss=correct=count=0
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(set_to_none=True); logits=model(x); loss=F.cross_entropy(logits,y); loss.backward()
            params=dict(model.named_parameters()); before={tag:params[name].detach().clone() for tag,name in TRACKED_PARAMETERS.items()}
            row={'run':run_name,'epoch':epoch,'step':global_step}
            for tag,name in TRACKED_PARAMETERS.items(): row[f'{tag}_grad_norm']=params[name].grad.detach().norm().item()
            optimizer.step(); params=dict(model.named_parameters())
            for tag,name in TRACKED_PARAMETERS.items():
                p=params[name]; delta=(p.detach()-before[tag]).norm().item(); row[f'{tag}_update_to_weight']=delta/(p.detach().norm().item()+1e-12)
                writer.add_scalar(f'gradient/{tag}',row[f'{tag}_grad_norm'],global_step); writer.add_scalar(f'update_to_weight/{tag}',row[f'{tag}_update_to_weight'],global_step)
            dynamics.append(row); global_step+=1; train_loss+=loss.item()*y.numel(); correct+=(logits.argmax(1)==y).sum().item(); count+=y.numel()
        val_loss,val_acc=evaluate(model,val_loader); hist={'run':run_name,'epoch':epoch,'train_loss':train_loss/count,'train_accuracy':correct/count,'val_loss':val_loss,'val_accuracy':val_acc}; history.append(hist); print(run_name,hist)
        writer.add_scalar('metric/val_loss',val_loss,epoch); writer.add_scalar('metric/val_accuracy',val_acc,epoch)
        if epoch in DIAG_EPOCHS: torch.save(model.state_dict(),LOCAL_CKPT_DIR/f'{run_name}_epoch{epoch}.pt')
    writer.close(); h=pd.DataFrame(history); d=pd.DataFrame(dynamics); h.to_csv(CSV_DIR/f'{run_name}_history.csv',index=False); d.to_csv(CSV_DIR/f'{run_name}_dynamics.csv',index=False); return model,h,d

sgd_model,sgd_history,sgd_dynamics=train_model('sgd','sgd')
adamw_model,adamw_history,adamw_dynamics=train_model('adamw','adamw')


## 4. 고정 입력 representation 추출


In [ ]:
def load_checkpoint(run,epoch): return torch.load(LOCAL_CKPT_DIR/f'{run}_epoch{epoch}.pt',map_location='cpu')
@torch.no_grad()
def collect_features(state,loader,max_samples=2000):
    model=SmallCNN().to(DEVICE); model.load_state_dict(state); model.eval(); buckets={k:[] for k in LAYER_NAMES}; labels=[]; seen=0
    for x,y in loader:
        x=x.to(DEVICE); _,f=model(x,return_features=True); take=min(x.size(0),max_samples-seen)
        for k in LAYER_NAMES: buckets[k].append(f[k][:take].cpu())
        labels.append(y[:take].cpu()); seen+=take
        if seen>=max_samples: break
    return {k:torch.cat(v).numpy() for k,v in buckets.items()},torch.cat(labels).numpy()
FEATURE_CACHE={}
for run in ['sgd','adamw']:
    for epoch in DIAG_EPOCHS: FEATURE_CACHE[(run,epoch)]=collect_features(load_checkpoint(run,epoch),train_eval_loader)
print('feature snapshots:',list(FEATURE_CACHE))


## 5. Effective rank + Linear probe


In [ ]:
def effective_rank(x):
    x=x-x.mean(0,keepdims=True); s=np.linalg.svd(x,compute_uv=False); lam=s*s; p=lam/(lam.sum()+1e-12); return float(np.exp(-(p*np.log(p+1e-12)).sum()))
rows=[]
for (run,epoch),(features,labels) in FEATURE_CACHE.items():
    for layer,x in features.items(): rows.append({'run':run,'epoch':epoch,'layer':layer,'effective_rank':effective_rank(x)})
rank_df=pd.DataFrame(rows); rank_df.to_csv(CSV_DIR/'effective_rank.csv',index=False); display(rank_df)
probe_rows=[]
for epoch in DIAG_EPOCHS:
    train_f,train_y=FEATURE_CACHE[('sgd',epoch)]
    for layer in LAYER_NAMES:
        x=train_f[layer]; cut=int(len(x)*0.8); scaler=StandardScaler().fit(x[:cut]); clf=LogisticRegression(max_iter=500,n_jobs=-1).fit(scaler.transform(x[:cut]),train_y[:cut]); acc=clf.score(scaler.transform(x[cut:]),train_y[cut:]); probe_rows.append({'epoch':epoch,'layer':layer,'probe_accuracy':acc})
probe_df=pd.DataFrame(probe_rows); probe_df.to_csv(CSV_DIR/'linear_probe.csv',index=False); display(probe_df)


## 6. CKA + PCA/UMAP + Local PCA


In [ ]:
def linear_cka(x,y):
    x=x-x.mean(0,keepdims=True); y=y-y.mean(0,keepdims=True); xty=x.T@y; return float((xty*xty).sum()/np.sqrt(((x.T@x)**2).sum()*((y.T@y)**2).sum()+1e-12))
cka_rows=[]
for layer in LAYER_NAMES:
    base=FEATURE_CACHE[('sgd',0)][0][layer]
    for epoch in DIAG_EPOCHS: cka_rows.append({'layer':layer,'epoch':epoch,'cka_to_init':linear_cka(base,FEATURE_CACHE[('sgd',epoch)][0][layer])})
cka_df=pd.DataFrame(cka_rows); cka_df.to_csv(CSV_DIR/'cka_to_init.csv',index=False); display(cka_df)
features,labels=FEATURE_CACHE[('sgd',EPOCHS)]; x=features['penultimate']; pca_xy=PCA(n_components=2).fit_transform(x); umap_xy=umap.UMAP(n_components=2,n_neighbors=20,min_dist=0.1,random_state=SEED).fit_transform(x)
fig,axes=plt.subplots(1,2,figsize=(11,4)); axes[0].scatter(pca_xy[:,0],pca_xy[:,1],c=labels,s=7); axes[0].set_title('PCA'); axes[1].scatter(umap_xy[:,0],umap_xy[:,1],c=labels,s=7); axes[1].set_title('UMAP'); plt.tight_layout(); plt.savefig(FIG_DIR/'pca_umap.png',dpi=170); plt.show()
x=StandardScaler().fit_transform(features['penultimate']); neighbors=NearestNeighbors(n_neighbors=31).fit(x).kneighbors(return_distance=False); local_dimensions=[]
for idx in range(min(500,len(x))):
    evr=PCA().fit(x[neighbors[idx]]).explained_variance_ratio_; local_dimensions.append(int(np.searchsorted(np.cumsum(evr),0.90)+1))
local_dimensions=np.array(local_dimensions); np.save(NPZ_DIR/'local_pca_dim90.npy',local_dimensions); print('local PCA dim90 mean:',local_dimensions.mean())


## 7. Hessian top eigenvalue


In [ ]:
x_h,y_h=next(iter(val_loader)); x_h,y_h=x_h[:128].to(DEVICE),y_h[:128].to(DEVICE)
def hessian_top_eigenvalue(state,iters=12):
    model=SmallCNN().to(DEVICE); model.load_state_dict(state); model.eval(); params=[p for p in model.parameters() if p.requires_grad]; v=[torch.randn_like(p) for p in params]; norm=torch.sqrt(sum((z*z).sum() for z in v)); v=[z/norm for z in v]
    lam=0.0
    for _ in range(iters):
        loss=F.cross_entropy(model(x_h),y_h); g=torch.autograd.grad(loss,params,create_graph=True); gv=sum((a*b).sum() for a,b in zip(g,v)); hv=torch.autograd.grad(gv,params); norm=torch.sqrt(sum((z*z).sum() for z in hv)); v=[z/(norm+1e-12) for z in hv]; lam=float(sum((a*b).sum() for a,b in zip(v,hv)).detach().cpu())
    return lam
hessian_rows=[]
for condition,state in [('SGD init',load_checkpoint('sgd',0)),('SGD final',load_checkpoint('sgd',EPOCHS)),('AdamW final',load_checkpoint('adamw',EPOCHS))]: hessian_rows.append({'condition':condition,'lambda_max':hessian_top_eigenvalue(state)})
hessian_df=pd.DataFrame(hessian_rows); hessian_df.to_csv(CSV_DIR/'hessian_top_eigenvalue.csv',index=False); display(hessian_df)


## 8. Weight interpolation


In [ ]:
def interpolate_state(a,b,alpha):
    return {k:((1-alpha)*a[k]+alpha*b[k] if torch.is_floating_point(a[k]) else (a[k] if alpha<0.5 else b[k])) for k in a}
@torch.no_grad()
def interpolation_curve(a,b):
    model=SmallCNN().to(DEVICE); rows=[]
    for alpha in np.linspace(0,1,21): model.load_state_dict(interpolate_state(a,b,float(alpha))); loss,acc=evaluate(model,val_loader); rows.append({'alpha':float(alpha),'loss':loss,'accuracy':acc})
    return pd.DataFrame(rows)
middle_epoch=DIAG_EPOCHS[-2]; same=interpolation_curve(load_checkpoint('sgd',middle_epoch),load_checkpoint('sgd',EPOCHS)); same['path']='SGD middle to SGD final'; cross=interpolation_curve(load_checkpoint('sgd',EPOCHS),load_checkpoint('adamw',EPOCHS)); cross['path']='SGD final to AdamW final'; interpolation_df=pd.concat([same,cross],ignore_index=True); interpolation_df.to_csv(CSV_DIR/'weight_interpolation.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,part in interpolation_df.groupby('path'): axes[0].plot(part['alpha'],part['loss'],marker='o',label=name); axes[1].plot(part['alpha'],part['accuracy'],marker='o',label=name)
axes[0].set_title('Interpolation loss'); axes[1].set_title('Interpolation accuracy')
for ax in axes: ax.set_xlabel('alpha'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'weight_interpolation.png',dpi=170); plt.show()


## 9. TensorBoard와 최종 저장 확인


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deep_learning_diagnostics/tensorboard


In [ ]:
summary={'dataset_source':'Hugging Face uoft-cs/cifar10','epochs':EPOCHS,'diagnostic_epochs':DIAG_EPOCHS,'sgd_final_val_accuracy':float(sgd_history.iloc[-1]['val_accuracy']),'adamw_final_val_accuracy':float(adamw_history.iloc[-1]['val_accuracy']),'local_pca_dim90_mean':float(local_dimensions.mean()),'hessian':dict(zip(hessian_df['condition'],hessian_df['lambda_max'].astype(float)))}
with open(SUMMARY_DIR/'experiment_summary.json','w') as f: json.dump(summary,f,indent=2)
assert not any(DRIVE_ROOT.rglob('*.pt')); print(json.dumps(summary,indent=2)); print('Storage policy passed: no .pt model weights on Drive.')


## 결과를 읽는 권장 순서

`loss/accuracy → gradient + update-to-weight → effective rank + probe → CKA → PCA/UMAP → local PCA → Hessian → interpolation`
